# 03 — DQN with Symbolic Grid Observations
Train a DQN agent using a tile-based symbolic grid extracted from the game screen.

In [ ]:
# --- Google Colab Setup ---
import os

if os.getenv('COLAB_RELEASE_TAG'):
    !pip install -q Cython setuptools wheel
    !git clone https://github.com/lmartim4/CSC-52081-ContinousMountainCar.git /content/repo
    %cd /content/repo
    !pip install -e ".[notebook]" --no-build-isolation
    # Apply NumPy 2.x / API compatibility patches
    import site; SITE = site.getsitepackages()[0]
    !patch -p0 {SITE}/nes_py/_rom.py                   < patches/nes_py_numpy2.patch
    !patch -p0 {SITE}/gym/utils/passive_env_checker.py < patches/gym_bool8_numpy2.patch
    !patch -p0 {SITE}/gym_super_mario_bros/smb_env.py  < patches/smb_env_numpy2.patch
    !patch -p1 {SITE}/gym/wrappers/time_limit.py       < patches/gym_time_limit_compat.patch
else:
    if os.path.basename(os.getcwd()) == 'notebooks':
        %cd ..

In [ ]:
# Create environment
env = make_symbolic_env(flatten=True)
print(f'Observation space: {env.observation_space.shape}')
print(f'Action space: {env.action_space.n}')

In [ ]:
# Configure for MLP policy (symbolic input is a flat vector)
config = DQNConfig(total_timesteps=2_000_000, policy='MlpPolicy')
model = make_dqn_agent(env, config=config, tensorboard_log='../logs/symbolic_dqn')

In [ ]:
# Train
callback = CheckpointAndLogCallback(
    save_path='../models/symbolic_dqn',
    save_freq=100_000,
)
model.learn(total_timesteps=config.total_timesteps, callback=callback)
model.save('../models/symbolic_dqn/final_model')

In [ ]:
# Quick test
from src.utils.evaluation import evaluate_agent
results = evaluate_agent(model, env, n_episodes=10)
print(f"Mean reward: {results['mean_reward']:.1f} ± {results['std_reward']:.1f}")
print(f"Flag rate: {results['flag_rate']:.2%}")